In [1]:
import sqlite3

conn = sqlite3.connect("trade_pipeline.db")
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON")

In [2]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS users (
    user_id INTEGER PRIMARY KEY,
    name TEXT
);
""")

conn.commit()

In [3]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS assets (
    asset_id INTEGER PRIMARY KEY,
    symbol TEXT,
    asset_type TEXT
);
""")

conn.commit()

In [4]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS orders (
    order_id INTEGER PRIMARY KEY,
    user_id INTEGER,
    asset_id INTEGER,
    amount REAL,
    price REAL,
    date TEXT,
    foreign key (user_id) references users(user_id),
    foreign key (asset_id) references assets(asset_id)
);
""")

conn.commit()

In [5]:
cursor.execute("""
insert into users (name) 
values ('Patrik'),('Anna'), ('Erik');
""")

conn.commit()

In [6]:
cursor.execute("""
SELECT * FROM users;
""")

cursor.fetchall()

[(1, 'Patrik'),
 (2, 'Anna'),
 (3, 'Erik'),
 (4, 'Patrik'),
 (5, 'Anna'),
 (6, 'Erik')]

In [7]:
cursor.execute("""
insert into assets (symbol, asset_type)
values ('AAPL', 'stock'), ('GOOGL', 'stock'), ('BTC', 'crypto');
""")

conn.commit()

In [8]:
cursor.execute("""
SELECT * FROM assets;
""")    
cursor.fetchall()

[(1, 'AAPL', 'stock'),
 (2, 'GOOGL', 'stock'),
 (3, 'BTC', 'crypto'),
 (4, 'AAPL', 'stock'),
 (5, 'GOOGL', 'stock'),
 (6, 'BTC', 'crypto')]

In [9]:
cursor.execute("""
insert into orders (user_id, asset_id, amount, price, date)
values (1, 1, 10, 150.0, '2024-06-01'), 
       (2, 2, 5, 2800.0, '2024-06-02'), 
       (3, 3, 0.5, 30000.0, '2024-06-03');

""")
conn.commit()

In [10]:
cursor.execute("""
SELECT * FROM orders;
""")

print(cursor.fetchall())

[(1, 1, 1, 10.0, 150.0, '2024-06-01'), (2, 2, 2, 5.0, 2800.0, '2024-06-02'), (3, 3, 3, 0.5, 30000.0, '2024-06-03'), (4, 1, 1, 10.0, 150.0, '2024-06-01'), (5, 2, 2, 5.0, 2800.0, '2024-06-02'), (6, 3, 3, 0.5, 30000.0, '2024-06-03')]


In [11]:
cursor.execute("""
SELECT
    u.name,
    a.symbol,
    o.amount * o.price AS total_value
FROM orders o
JOIN users u ON o.user_id = u.user_id
JOIN assets a ON o.asset_id = a.asset_id;
""")
print(cursor.fetchall())

[('Patrik', 'AAPL', 1500.0), ('Anna', 'GOOGL', 14000.0), ('Erik', 'BTC', 15000.0), ('Patrik', 'AAPL', 1500.0), ('Anna', 'GOOGL', 14000.0), ('Erik', 'BTC', 15000.0)]


In [12]:
cursor.execute("""
SELECT
    u.name,
    SUM(o.amount * o.price) AS total_traded_value
FROM orders o
JOIN users u ON o.user_id = u.user_id
GROUP BY u.user_id, u.name;
    
""")
print(cursor.fetchall())

[('Patrik', 3000.0), ('Anna', 28000.0), ('Erik', 30000.0)]


In [13]:
cursor.execute("""
SELECT
    u.name,
    a.symbol,
    SUM(o.amount * o.price) AS total_traded_value
FROM orders o
JOIN users u ON o.user_id = u.user_id
JOIN assets a ON o.asset_id = a.asset_id
GROUP BY u.user_id, u.name, a.symbol
ORDER BY total_traded_value DESC    
""")
print(cursor.fetchall())

[('Erik', 'BTC', 30000.0), ('Anna', 'GOOGL', 28000.0), ('Patrik', 'AAPL', 3000.0)]


In [22]:
cursor.execute("""
SELECT
    u.name,
    COUNT(*) AS total_orders,
    SUM(o.amount * o.price) AS total_traded_value,
    AVG(o.amount * o.price) AS avg_order_value
FROM orders o
JOIN users u ON o.user_id = u.user_id
GROUP BY u.user_id, u.name
ORDER BY total_traded_value DESC
""")
print(cursor.fetchall())

[('Erik', 2, 30000.0, 15000.0), ('Anna', 2, 28000.0, 14000.0), ('Patrik', 2, 3000.0, 1500.0)]
